In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import seaborn as sns
import numpy as np

warnings.filterwarnings("ignore")
sns.set(style="whitegrid", color_codes=True)
%matplotlib inline

# SALES INPUT

In [ ]:
data_sales = pd.read_excel("Training_Data.xlsx")
print(data_sales.shape)
data_sales.head()

In [ ]:
validratio = ~data_sales.isnull()
validratio = validratio.sum(axis=0).iloc[::-1]
validratio

# STORES INPUT

In [ ]:
data_stores = pd.read_csv("stores.csv")
print(data_stores.shape)
data_stores.head()

In [ ]:
validratio = ~data_stores.isnull()
validratio = validratio.sum(axis=0).iloc[::-1]
validratio = pd.DataFrame(validratio/data_stores.shape[0] * 100, columns=['Percentage of valid values'])
validratio

# FILL VALUES

In [ ]:
cd_mean = data_stores['CompetitionDistance'].mean()
data_stores['CompetitionDistance'] = data_stores['CompetitionDistance'].fillna(cd_mean)
cd_mean

# JOIN DATA

In [ ]:
data_train = data_sales.join(data_stores, 'Store', rsuffix='_OL', how='inner')
print(data_train.shape)
data_train.head()

In [ ]:
data_train = data_train.drop(['Store_OL'], axis=1)

# FEATURES CLEANING AND ENGINEERING

In [ ]:
data_train['Day'] = data_train['Date'].dt.day
data_train['Month'] = data_train['Date'].dt.month
data_train['Year'] = data_train['Date'].dt.month
data_train = data_train.drop(['DayOfWeek'], axis=1)

In [ ]:
# CompetitionOpenSince___ has irrelavent data
data_train = data_train.drop(['CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth'], axis=1)

In [ ]:
import datetime
from datetime import timedelta

def promo3(x):
    if x['Promo2'] == 0:
        return False
    if pd.isna(x['Promo2SinceYear']) or pd.isna(x['PromoInterval']):
        return False
    
    promoDt = datetime.date(int(x['Promo2SinceYear']), 1, 1)
    if not pd.isna(x['Promo2SinceWeek']):
        promoDt += timedelta(weeks=x['Promo2SinceWeek'])
    
    if promoDt <= x['Date'].date():
        return x['Date'].strftime('%b') in x['PromoInterval']
    else:
        return False

data_train['OnRecurringPromo'] = data_train.apply(promo3, axis=1)

In [ ]:
data_train = data_train.drop(['Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval'], axis=1)

In [ ]:
data_train.head()

# Normalize Values

In [ ]:
data_train['Customers_n'] = (data_train['Customers'] - data_train['Customers'].mean())/data_train['Customers'].std()
data_train['CompetitionDistance_n'] = (data_train['CompetitionDistance'] - data_train['CompetitionDistance'].mean())/data_train['CompetitionDistance'].std()

In [ ]:
data_train.head()

# Save as trainable format

In [ ]:
inputSet = np.zeros((data_train.shape[0], 19), dtype=np.float32)
outputSet = np.zeros(data_train.shape[0], dtype=np.float32)
#idxToId = np.zeros(data_train.shape[0], dtype=np.int32)
for idx, x in enumerate(data_train.iterrows()):
    row = x[1]
    
    #idxToId[idx] = row['PassengerId']
    outputSet[idx] = row['Sales']
        
    inputSet[idx][0] = row['Customers_n']
    inputSet[idx][1] = row['Open']
    
    if row['StateHoliday'] == 'a': inputSet[idx][2] = 1
    elif row['StateHoliday'] == 'b': inputSet[idx][3] = 1
    elif row['StateHoliday'] == 'c': inputSet[idx][4] = 1
        
    inputSet[idx][5] = row['SchoolHoliday']
    
    if row['StoreType'] == 'a': inputSet[idx][6] = 1
    elif row['StoreType'] == 'b': inputSet[idx][7] = 1
    elif row['StoreType'] == 'c': inputSet[idx][8] = 1
    elif row['StoreType'] == 'd': inputSet[idx][9] = 1
    
    if row['Assortment'] == 'a': inputSet[idx][10] = 1
    elif row['Assortment'] == 'b': inputSet[idx][11] = 1
    elif row['Assortment'] == 'c': inputSet[idx][12] = 1
        
    inputSet[idx][13] = row['Promo']
    inputSet[idx][14] = row['OnRecurringPromo']
    inputSet[idx][15] = row['Day']/31
    inputSet[idx][16] = row['Month']/12
    inputSet[idx][17] = row['CompetitionDistance_n']

In [ ]:
np.savez("dataset.npz", inputSet=inputSet, outputSet=outputSet)

In [ ]:
np.sum(inputSet, axis=0).astype(np.int32)